# Qwen embedding submission diagnostic

Runs the exact submission preprocessing on the saved component-disjoint validation and compares it with the original Kaggle predictions.

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "catboost==1.2.8", "rapidfuzz==3.14.1", "transformers==4.57.6"
], check=True)


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working/submission_diagnostic")
WORK.mkdir(parents=True, exist_ok=True)

def one(candidates, label):
    candidates = list(candidates)
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one {label}, found {len(candidates)}: {candidates[:20]}")
    print(f"{label}: {candidates[0]}")
    return candidates[0]

items_path = one(INPUT.rglob("items_human.parquet"), "items")
human_matches_path = one(
    (p for p in INPUT.rglob("matches.parquet") if p.name == "matches.parquet"),
    "human matches",
)
saved_validation_path = one(
    (
        p for p in INPUT.rglob("validation_predictions.parquet")
        if p.parent.name == "03_names_qwen_attributes"
    ),
    "saved validation predictions",
)
catboost_path = one(
    (p for p in INPUT.rglob("model.cbm") if p.parent.name == "03_names_qwen_attributes"),
    "CatBoost model",
)
keys_path = one(
    (
        p for p in INPUT.rglob("selected_attribute_keys.json")
        if "embedding_boosting" in str(p)
    ),
    "selected attribute keys",
)
qwen_weights = one(
    (
        p for p in INPUT.rglob("model.safetensors")
        if p.parent.name == "qwen_embedding_model" and p.stat().st_size > 1_000_000_000
    ),
    "Qwen weights",
)
qwen_dir = qwen_weights.parent

print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("Qwen bytes:", qwen_weights.stat().st_size)


In [ ]:
runner_source = "#!/usr/bin/env python3\n\"\"\"Deadline-aware offline Qwen item embeddings + structured CatBoost submission.\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport os\nimport re\nimport sys\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\nos.environ.setdefault(\"HF_HUB_OFFLINE\", \"1\")\nos.environ.setdefault(\"TRANSFORMERS_OFFLINE\", \"1\")\nos.environ.setdefault(\"TOKENIZERS_PARALLELISM\", \"true\")\nos.environ.setdefault(\"OMP_NUM_THREADS\", \"20\")\n\nimport numpy as np\nimport pandas as pd\nfrom rapidfuzz import fuzz\n\n\nROOT = Path(__file__).resolve().parent\nMODEL_DIR = Path(os.getenv(\"PM_MODEL_DIR\", \"/opt/models/qwen3-embedding-0.6b\"))\nif not MODEL_DIR.is_dir():\n    MODEL_DIR = ROOT / \"models\" / \"qwen_embedding_model\"\nCATBOOST_PATH = ROOT / \"models\" / \"matching_model.cbm\"\nKEYS_PATH = ROOT / \"selected_attribute_keys.json\"\nMAX_LENGTH = int(os.getenv(\"PM_MAX_LENGTH\", \"96\"))\nEMBEDDING_DIMENSION = 256\nBATCH_SIZE = int(os.getenv(\"PM_BATCH_SIZE\", \"2048\"))\nPUBLIC_PAIR_BOUNDARY = int(os.getenv(\"PM_PUBLIC_PAIR_BOUNDARY\", \"250000\"))\nPUBLIC_SOFT_LIMIT = float(os.getenv(\"PM_PUBLIC_SOFT_LIMIT_SECONDS\", \"325\"))\nPRIVATE_SOFT_LIMIT = float(os.getenv(\"PM_PRIVATE_SOFT_LIMIT_SECONDS\", \"745\"))\nSPACE_RE = re.compile(r\"\\s+\")\nNUMBER_RE = re.compile(r\"\\d+(?:[.,]\\d+)?\")\nFAMILY_PATTERNS = {\n    \"brand\": re.compile(r\"бренд|brand|производител\"),\n    \"model\": re.compile(r\"модель|model|серия|линейка\"),\n    \"identifier\": re.compile(r\"артикул|партномер|part.?number|sku|mpn|oem|код товара\"),\n    \"size\": re.compile(r\"размер|длина|ширина|высота|диаметр|толщина|габарит\"),\n    \"quantity\": re.compile(r\"количеств|комплект|упаков|штук|шт\\.?$|объ.м|вес\"),\n    \"color\": re.compile(r\"цвет|оттенок\"),\n    \"material\": re.compile(r\"материал|состав|сырь\"),\n    \"country\": re.compile(r\"страна|производств\"),\n    \"seller_noise\": re.compile(r\"продав|магазин|поставщик|валюта|цена|достав|гарант\"),\n}\n\n\ndef parse_args():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--items_path\", \"--items-path\", \"-i\", required=True, type=Path)\n    parser.add_argument(\"--matches_path\", \"--matches-path\", \"-m\", required=True, type=Path)\n    parser.add_argument(\"--output_path\", \"--output-path\", \"-o\", required=True, type=Path)\n    parser.add_argument(\"--limit\", type=int)\n    parser.add_argument(\"--skip-embeddings\", action=\"store_true\")\n    return parser.parse_args()\n\n\ndef log(message, started): print(f\"[{time.perf_counter()-started:7.1f}s] {message}\", flush=True)\n\n\ndef clean(value):\n    if value is None: return \"\"\n    if isinstance(value, (dict, list)): value = json.dumps(value, ensure_ascii=False, sort_keys=True)\n    return SPACE_RE.sub(\" \", str(value)).strip().casefold().replace(\"ё\", \"е\")\n\n\ndef parse_attributes(raw):\n    if not isinstance(raw, str) or not raw: return {}\n    value = json.loads(raw)\n    if not isinstance(value, dict): return {}\n    result = {}\n    for key, item in value.items():\n        key, item = clean(key), clean(item)\n        if key and item: result[key] = item\n    return result\n\n\ndef family_for_key(key):\n    for family, pattern in FAMILY_PATTERNS.items():\n        if pattern.search(key): return family\n    return None\n\n\ndef load_data(args):\n    matches = pd.read_parquet(args.matches_path, columns=[\"id1\", \"id2\"])\n    if args.limit: matches = matches.head(args.limit).copy()\n    required = pd.unique(matches[[\"id1\", \"id2\"]].to_numpy().reshape(-1))\n    items = pd.read_parquet(args.items_path, columns=[\"id\", \"name\", \"attributes\", \"category\"])\n    items = items[items.id.isin(required)].reset_index(drop=True)\n    if len(items) != len(required): raise ValueError(\"matches reference missing item IDs\")\n    lookup = pd.Series(np.arange(len(items), dtype=np.int32), index=items.id.to_numpy())\n    left = lookup.loc[matches.id1].to_numpy(); right = lookup.loc[matches.id2].to_numpy()\n    categories = items.category.astype(str).to_numpy()[left]\n    return matches, items, left, right, categories\n\n\ndef name_features(names, left, right, categories):\n    rows = []\n    for lp, rp in zip(left, right):\n        first, second = clean(names[lp]), clean(names[rp])\n        fn, sn = set(NUMBER_RE.findall(first)), set(NUMBER_RE.findall(second)); union = fn | sn\n        longest = max(len(first), len(second))\n        rows.append((fuzz.ratio(first,second)/100, fuzz.token_set_ratio(first,second)/100,\n                     fuzz.token_sort_ratio(first,second)/100, float(first==second),\n                     min(len(first),len(second))/longest if longest else 1.0,\n                     len(fn&sn)/max(1,len(union)), float(bool(fn) and bool(sn)), abs(len(first)-len(second))))\n    frame = pd.DataFrame(rows, columns=[\"name_ratio\",\"name_token_set_ratio\",\"name_token_sort_ratio\",\"name_exact\",\"name_length_ratio\",\"name_numeric_jaccard\",\"name_numbers_both\",\"name_length_delta\"], dtype=np.float32)\n    return pd.concat([frame, pd.get_dummies(pd.Series(categories,name=\"category\"),prefix=\"category\",dtype=np.float32)], axis=1)\n\n\ndef attribute_features(categories, left, right, attrs, selected, per_category=5):\n    rows=[]; families=list(FAMILY_PATTERNS)\n    for category,lp,rp in zip(categories,left,right):\n        first,second=attrs[lp],attrs[rp]; fk,sk=set(first),set(second); common=fk&sk; union=fk|sk\n        equal=sum(first[k]==second[k] for k in common); conflict=len(common)-equal\n        ff,sf=defaultdict(set),defaultdict(set)\n        for k,v in first.items(): ff[family_for_key(k)].add(v)\n        for k,v in second.items(): sf[family_for_key(k)].add(v)\n        row=[len(common),len(common)/max(1,len(union)),equal,conflict,equal/max(1,len(common)),conflict/max(1,len(common)),abs(len(first)-len(second)),float(first==second)]\n        for family in families:\n            a,b=ff[family],sf[family]; row += [float(bool(a and b)),float(bool(a&b)),float(bool(a and b and not(a&b)))]\n        keys=selected.get(str(category),[])\n        for rank in range(per_category):\n            key=keys[rank] if rank<len(keys) else None; both=bool(key and key in first and key in second)\n            row += [float(both),float(both and first[key]==second[key]),float(both and first[key]!=second[key]),float(bool(key) and ((key in first)!=(key in second)))]\n        rows.append(row)\n    columns=[\"attr_shared_keys\",\"attr_key_jaccard\",\"attr_equal_values\",\"attr_conflicting_values\",\"attr_equal_ratio\",\"attr_conflict_ratio\",\"attr_count_delta\",\"attr_exact\"]\n    for family in families: columns += [f\"{family}_both\",f\"{family}_match\",f\"{family}_conflict\"]\n    for rank in range(per_category): columns += [f\"exact_key_{rank}_both\",f\"exact_key_{rank}_match\",f\"exact_key_{rank}_conflict\",f\"exact_key_{rank}_one_missing\"]\n    return pd.DataFrame(rows,columns=columns,dtype=np.float32)\n\n\ndef last_token_pool(hidden, mask, torch):\n    lengths=mask.sum(dim=1)-1\n    return hidden[torch.arange(hidden.shape[0],device=hidden.device),lengths]\n\n\ndef encode_names(names, deadline, started):\n    import torch\n    import torch.nn.functional as F\n    from transformers import AutoModel, AutoTokenizer\n    tokenizer=AutoTokenizer.from_pretrained(MODEL_DIR,local_files_only=True,padding_side=\"left\")\n    model=AutoModel.from_pretrained(MODEL_DIR,local_files_only=True,torch_dtype=torch.bfloat16,attn_implementation=\"sdpa\").cuda().eval()\n    result=np.zeros((len(names),EMBEDDING_DIMENSION),dtype=np.float16); completed=0\n    with torch.inference_mode(), torch.autocast(\"cuda\",dtype=torch.bfloat16):\n        for start in range(0,len(names),BATCH_SIZE):\n            if time.perf_counter()>=deadline: break\n            end=min(len(names),start+BATCH_SIZE)\n            batch=tokenizer(names[start:end],padding=True,truncation=True,max_length=MAX_LENGTH,pad_to_multiple_of=8,return_tensors=\"pt\")\n            batch={k:v.cuda(non_blocking=True) for k,v in batch.items()}\n            embedding=last_token_pool(model(**batch).last_hidden_state,batch[\"attention_mask\"],torch)[:,:EMBEDDING_DIMENSION]\n            embedding=F.normalize(embedding.float(),p=2,dim=1)\n            result[start:end]=embedding.cpu().numpy().astype(np.float16); completed=end\n            if completed%50000<BATCH_SIZE: log(f\"Embedded {completed:,}/{len(names):,} items\",started)\n    return result,completed\n\n\ndef embedding_features(embeddings,left,right):\n    first=np.asarray(embeddings[left],np.float32); second=np.asarray(embeddings[right],np.float32)\n    absolute=np.abs(first-second); product=first*second\n    data={\"embedding_cosine\":np.einsum(\"ij,ij->i\",first,second),\"embedding_l1_mean\":absolute.mean(1),\"embedding_l2\":np.sqrt(np.square(first-second).sum(1)),\"embedding_abs_max\":absolute.max(1)}\n    for i in range(EMBEDDING_DIMENSION): data[f\"embedding_abs_{i:03d}\"]=absolute[:,i]; data[f\"embedding_product_{i:03d}\"]=product[:,i]\n    return pd.DataFrame(data,dtype=np.float32)\n\n\ndef align(frame, feature_names):\n    missing=set(feature_names)-set(frame.columns)\n    for column in missing: frame[column]=np.float32(0)\n    return frame.loc[:,feature_names]\n\n\ndef fallback_scores(lexical):\n    return (0.7*lexical.name_numeric_jaccard+0.3*lexical.name_token_set_ratio).to_numpy(np.float32)\n\n\ndef main():\n    args=parse_args(); started=time.perf_counter(); matches,items,left,right,categories=load_data(args)\n    soft=PUBLIC_SOFT_LIMIT if len(matches)<=PUBLIC_PAIR_BOUNDARY else PRIVATE_SOFT_LIMIT; deadline=started+soft\n    log(f\"Loaded {len(matches):,} pairs and {len(items):,} required items; soft limit={soft:.0f}s\",started)\n    names=items.name.fillna(\"\").astype(str).to_numpy(); lexical=name_features(names,left,right,categories)\n    attrs=[parse_attributes(raw) for raw in items.attributes]; selected=json.loads(KEYS_PATH.read_text(encoding=\"utf-8\")); structured=attribute_features(categories,left,right,attrs,selected)\n    fallback=fallback_scores(lexical); scores=fallback.copy()\n    if not args.skip_embeddings:\n        embeddings,completed=encode_names(names.tolist(),deadline-30,started)\n        if completed==len(items) and time.perf_counter()<deadline-15:\n            from catboost import CatBoostClassifier\n            model=CatBoostClassifier(); model.load_model(CATBOOST_PATH)\n            features=pd.concat([lexical,embedding_features(embeddings,left,right),structured],axis=1)\n            scores=model.predict_proba(align(features,model.feature_names_))[:,1]\n            log(\"Applied Qwen + attributes CatBoost to every pair\",started)\n        else: log(f\"Embedding deadline reached at {completed:,}/{len(items):,}; using full lexical fallback\",started)\n    if len(scores)!=len(matches) or not np.isfinite(scores).all(): raise RuntimeError(\"invalid scores\")\n    output=matches[[\"id1\",\"id2\"]].copy(); output[\"predict\"]=scores\n    args.output_path.parent.mkdir(parents=True,exist_ok=True); output.to_csv(args.output_path,index=False)\n    log(f\"Saved {len(output):,} rows\",started); return 0\n\n\nif __name__==\"__main__\": raise SystemExit(main())\n"
runner_path = WORK / "run.py"
runner_path.write_text(runner_source, encoding="utf-8")
(WORK / "models").mkdir(exist_ok=True)
shutil.copy2(catboost_path, WORK / "models" / "matching_model.cbm")
shutil.copy2(keys_path, WORK / "selected_attribute_keys.json")

truth = pd.read_parquet(saved_validation_path)
required = {"id1", "id2", "target", "category", "predict"}
if not required.issubset(truth.columns):
    raise RuntimeError(f"Saved validation lacks columns: {required - set(truth.columns)}")
matches_path = WORK / "validation_pairs.parquet"
truth[["id1", "id2"]].to_parquet(matches_path, index=False)
print("Validation pairs:", len(truth), "positive rate:", float(truth.target.mean()))


In [ ]:
output_path = WORK / "submission_predictions.csv"
env = os.environ.copy()
env.update({
    "PM_MODEL_DIR": str(qwen_dir),
    "PM_BATCH_SIZE": "256",
    "PM_PUBLIC_SOFT_LIMIT_SECONDS": "7200",
    "PM_PRIVATE_SOFT_LIMIT_SECONDS": "7200",
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
})
command = [
    sys.executable, "-u", str(runner_path),
    "--items_path", str(items_path),
    "--matches_path", str(matches_path),
    "--output_path", str(output_path),
]
started = time.perf_counter()
result = subprocess.run(command, env=env, text=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT)
elapsed = time.perf_counter() - started
print(result.stdout)
(WORK / "diagnostic.log").write_text(result.stdout, encoding="utf-8")
if result.returncode:
    raise RuntimeError(f"Submission runner failed with code {result.returncode}")


In [ ]:
submitted = pd.read_csv(output_path)
if not np.array_equal(submitted[["id1", "id2"]].to_numpy(), truth[["id1", "id2"]].to_numpy()):
    raise RuntimeError("Submission changed validation pair order")

def macro_ap(frame, score_column):
    per_category = {
        str(category): float(average_precision_score(group.target, group[score_column]))
        for category, group in frame.groupby("category", sort=True)
    }
    return float(np.mean(list(per_category.values()))), per_category

comparison = truth[["id1", "id2", "target", "category", "predict"]].rename(
    columns={"predict": "training_pipeline_predict"}
)
comparison["submission_pipeline_predict"] = submitted.predict.to_numpy()
submission_macro, per_category = macro_ap(comparison, "submission_pipeline_predict")
training_macro, _ = macro_ap(comparison, "training_pipeline_predict")
overall = float(average_precision_score(comparison.target, comparison.submission_pipeline_predict))
pearson = float(comparison[["training_pipeline_predict", "submission_pipeline_predict"]].corr().iloc[0, 1])
spearman = float(comparison[["training_pipeline_predict", "submission_pipeline_predict"]].corr(method="spearman").iloc[0, 1])
mae = float(np.mean(np.abs(comparison.training_pipeline_predict - comparison.submission_pipeline_predict)))

report = {
    "pairs": len(comparison),
    "required_items": int(pd.unique(comparison[["id1", "id2"]].to_numpy().reshape(-1)).size),
    "submission_macro_average_precision": submission_macro,
    "training_pipeline_macro_average_precision": training_macro,
    "submission_overall_average_precision": overall,
    "prediction_pearson": pearson,
    "prediction_spearman": spearman,
    "prediction_mae": mae,
    "total_runner_seconds": elapsed,
    "per_category_average_precision": per_category,
    "runner_log": result.stdout.splitlines(),
    "qwen_model_directory": str(qwen_dir),
    "qwen_weights_bytes": qwen_weights.stat().st_size,
}
comparison.to_parquet(WORK / "prediction_comparison.parquet", index=False)
(WORK / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
(WORK / "COMPLETED").write_text("ok\n", encoding="utf-8")
print(json.dumps(report, ensure_ascii=False, indent=2))
